# Visualizing Kaggle Dataset in Python

In this notebook we will be learning how to download Kaggle datasets into a Python scripts to be able to work with the data and represent them in a graph. To do this we will be using the `kagglehub` library to get the datasets and the `pandas` library to work with them. For the graph representation we will be using `pyvis`, a Python library that creates interactive HTML graphs easily.

## 1. Import data

In the Kaggle webpage, there is a _code_ tab that gives you the code to import the data directly on a Python script using the `kagglehub` library. Using this allows you to use the dataset without having to download it manually on the computer and adding it to the repository or workplace.

Problem is, using this code I get errors reading the files I cannot fix. Because of this I opted to look for another way to import the data reading the `kagglehub` documentation and get the paths using the `os` library.

> On this part I also got AI support to understand the functions of the `kagglehub` library and the usage of the `os` library to get the paths of the files.

After obtaining the paths of the _.csv_ files containing the data, we are going to open this files as pandas dataframes to be able to work with them.


So, first thing we need to do is to import the libraries we are going to use to import this data:

In [2]:
import os
import pandas as pd
import kagglehub

Then, we download the whole dataset from Kaggle:

In [3]:
handle = "jfreyberg/spotify-artist-feature-collaboration-network"
dataset_path = kagglehub.dataset_download(handle, force_download=True) # download the whole dataset

100%|██████████| 14.6M/14.6M [00:01<00:00, 15.1MB/s]

Extracting files...


> The _handle_ is __kaggleUsername/datasetName__ that can be found on the kaggle URL or even in this code tab we were talking before. In this case, we are working with a dataset that shows the collaborations between artists.

> The _force_download_ parameter is necessary because sometimes the data will not be downloaded if found on the cache and can give some trouble.

Now, we have the dataset downloaded in our computer. With the 'os' library we are going to recover the individual paths of the two files that make up this dataset.

In [4]:
nodes_path = os.path.join(dataset_path, "nodes.csv") # inner path to nodes.csv
edges_path = os.path.join(dataset_path, "edges.csv") # inner path to edges.csv

Finally, we use pandas to read both _.csv_ and get all the data as frames:

In [5]:
df_nodes = pd.read_csv(nodes_path)
df_edges = pd.read_csv(edges_path)

## 2. Prepare the data

In this case, the dataset is pretty big (approximately 156.320 rows). From a data analysis perspective, a large dataset is ideal for working with and studying the different relationships between nodes. However, we are currently in a learning phase, so a large dataset will only make the algorithm take much longer to complete (and Python is not the most efficient language for this task).

Because of this, we are going to trim the dataset to _n_ nodes as follows:

In [6]:
n = 300
trim_nodes = df_nodes.head(n)
valid_nodes = set(trim_nodes["spotify_id"].astype(str))

> _valid_nodes_ is a set that contains all the nodes that are being consider on the study. Using a set will allow us to check if a node belongs to our subnetwork in constant complexity.

## 3. Generate the graph

To create the graph we are going to use the `pyvis` library because it is a library we have already used in other projects. Throughout this learning process, other libraries with a similar purpose will be explored to determine the best one for completing the final degree project.

First of all, we need to import the library, specifically the Network function to create the net we are going to add the nodes and edges :

In [7]:
from pyvis.network import Network

Then, we create the net adding all the nodes on _trim_nodes_ and adding and edge between nodes if both nodes are in the _valid_nodes_ set.

In [10]:
nt = Network(directed = False, notebook = True)
for i in range(0, n):
    node = trim_nodes.iloc[i]
    nt.add_node(node['spotify_id'], label = node['name'], color = 'blue')

for i in range(0, len(df_edges)):
    edge = df_edges.iloc[i]
    if edge['id_0'] in valid_nodes and edge['id_1'] in valid_nodes:
        nt.add_edge(edge['id_0'], edge['id_1'])

Finally, we generate the HTML file to show the interactive graph:

In [11]:
nt.show(name='kaggle-data-preview/test_graph.html', notebook=True)

kaggle-data-preview/test_graph.html


> The code script and the HTML file can be found on the _kaggle-data-preview_ directory

## 4. Modyfing the relations

Now that we have been able to import and show the Kaggle data, next step is modifying the relations as we like.
 This particular dataset from Kaggle models the collaborations between the artists, but it is not what we want to show in the project.

The Kaggle dataset has the _nodes.csv_ and the _edges.csv_. The idea is to keep using the _nodes.csv_ (because it has all the data we want to study) and build the edges manually given a condition.

So, in this notebook we already have the _nodes.csv_ imported. In this case, we are going to connect the artists whether they have at least one genre in common. So, before trimming the dataset, we have to set the genres attribute to a set, given that pandas reads the attributes as Strings.

In [12]:
import ast

df = pd.read_csv(nodes_path)
df["genres"] = df["genres"].apply(ast.literal_eval)

Now, we trim the dataset as before:

In [18]:
n = 300
trim_nodes = df.head(n)
valid_nodes = set(trim_nodes["spotify_id"].astype(str))

Now we have to establish when two nodes are connected. As explained before, this happens when the two nodes have at least one genre in common. To study this, we create the following function:

In [20]:
def are_connected(node_1, node_2):
    connected = False
    name = ''
    set_node_1 = node_1['genres']
    set_node_2 = node_2['genres']
    for genre in set_node_1:
        if genre in set_node_2:
            connected = True
            if name:
                name = name + '/ '
            name = name + str(genre)
    return connected, name

This function returns a boolean that indicates if the nodes are connected or not and a String indicating all the genres that both nodes have in common (this will later be used in the graph representation).

With this auxiliar function, we can now create the set of edges that are going to be connecting the nodes in our graph. This set keeps triplets composed by two nodes id and a String keeping all the information necessary to represent the connection.

In [21]:
edges = []
for i in range(0,n):
    for j in range(i+1,n): # graph not directed (given the chosen condition)
        node_1 = trim_nodes.iloc[i]
        node_2 = trim_nodes.iloc[j]
        connected, name = are_connected(node_1, node_2)
        if connected:
            edges.append((node_1['spotify_id'], node_2['spotify_id'], name))

Now, as before, we create the net with this new information:

In [22]:
nt = Network(directed = False, notebook = True)
for i in range(0, n):
    node = trim_nodes.iloc[i]
    nt.add_node(node['spotify_id'], label = node['name'], color = 'blue')

for node_1, node_2, name in edges:
    nt.add_edge(node_1, node_2, label = name)

Finally, we show the new graph:

In [23]:
nt.show(name='rewired-graph-generation/modified_graph.html', notebook=True)

modified_graph.html


> The code script and the HTML file can be found on the "_rewired-graph-generation_" directory in this repository. It is recommended to visualize the HTML on a navigator given that it allows to zoom in the graph to read the labels of the connections and nodes.